# 007 · Dilution model

**The question.** Will this company issue many new shares soon?

**The answer.** `dilution`: 1 if shares outstanding (split-adjusted, as first filed) rise by 10% or more within 182 days, else 0. About 10% of rows are 1.

**The baseline to beat.** the base rate: the same chance (about 9.6%) for every row.

**Measures.** Log loss (deciding, V2-37), AUC, Brier score and lift in the top tenth.

**Status.** Third version (5 November 2026): Jassin's notebook of 2 November (V2-40), brought up
to the rules decided since: the date-checked folds of `eda.folds`, every fitted step a `Dated`
model that knows when it learned (V2-49), the split-like answers dropped by code (V2-39, V2-40),
the year-by-year one-standard-error choice (V2-50) and a verdict per hypothesis (V2-46). Waiting
for the researcher's approval.

### What the answer means (read this first)

Think of an exam: the 96 **features** are the questions (what was known about a company on a
month-end: cash, losses, recent filings, price...), and **`dilution`** is the answer key (what
happened next). The model learns from rows *with* the key and is judged on rows whose key it never
saw.

- **The answer:** 1 if the company's shares outstanding (as filed on its 10-Q and 10-K covers,
  split-adjusted) rose **10% or more within the next 182 days** (6 months), 0 if not, missing if
  no new count was filed. Example: Company X on 31 March 2021 had 50 million shares (last count
  filed by then); the last count filed by 29 September 2021 says 58 million: +16%, so 1. With 52
  million (+4%), 0. With no new filing, missing (the row is not used).
- **Why 10%, not "any rise":** share counts creep up almost everywhere (employee stock: the median
  company +0.6% a year; two thirds of rows rise at all), so "any rise" would flag most healthy
  companies. What hurts shareholders is a share sale, which shows as a jump; above 10% the typical
  jump is +27%. The 10% is pre-registered (`config/risk_params.yaml` v5) and stays (decided 30
  Oct; an "offering event" question is roadmap 024).
- **Filings vs share counts:** filings (offering prospectuses, shelf registrations, 8-K 3.02
  unregistered sales) come early and are **features**: warning signs (after an offering filed in
  the past year, 21% of rows dilute 10%+ next; after an 8-K unregistered sale 27%; with none of
  these warning signs 2.3%). The share count is
  late but true, so it is the **answer**. The app shows the chance with these warning signs beside
  it.
- **Which rows and columns:** fit only on `dilution.parquet`, rows with `eligible_dilution` true
  and `dilution` present (409,576; base rate 9.6%); columns: the 96 features, never `dilution`
  or `window_end`. Save the feature list with the model.
- **How error is measured, in three stages, always on answers the model has not seen:** (1) the
  walk-forward test years 2019–2022 inside `dilution.parquet` (train on earlier years, score the
  year: this chooses the model); (2) the holdout, once (feature 010): the model predicts the
  2023+ month-end rows of `feature_snapshot.parquet`, which have **features but no answers** on
  purpose; the guarded command then computes their real answers from the share counts filed since
  and scores the predictions; (3) live (feature 009): each day's forecast is stored and scored
  six months later. `latest.parquet` is each company's newest row, for today's forecast.

### Rules for every model notebook

- **Spec first.** Before filling this notebook, run `/speckit-specify` for this feature with
  `SPECIFY_FEATURE_DIRECTORY=specs/007-dilution-model` on branch `007-dilution-model` (README, "Working
  together").
- **This notebook is where the model is built.** Every level is created, run and tuned here,
  with its tables and charts, start to finish by one command, so the researcher can read how the
  choice was made. Once the researcher is happy with the chosen model, its code moves into
  `src/sentinel/models/` (training, scoring, calibration) so that `sentinel update` scores every
  company each night without a notebook and the repository runs from a clone; this notebook then
  imports that code and must give the same numbers (docs/plan.md, "Notebooks and code").
- **Data.** Read only `data/exports/ml/dilution.parquet` through `sentinel.eda.load("dilution")`, which keeps
  the development rows (dated 2022 or earlier). Learn and test only on rows where this question's
  flag `eligible_dilution` is true and the answer is present (V2-26, V2-27). Never read `data/exports/reference/` (answers for 2023 on) and
  never compute anything for 2023 or later: that is the sealed holdout, tested once by feature 010
  (constitution IV).
- **Never use as features:** `dilution`, `window_end`, `cik`, `as_of` and the flags.
- **Never a random train/test split.** Rows of one company over time are not independent, and a
  random split lets the model learn from the future. The only split is walk-forward by year:
- **Evaluation is fixed** (decision D60, `config/risk_params.yaml` v5, never edited): yearly
  walk-forward test years 2019, 2020, 2021, 2022; for each, train only on rows whose answer window
  ended before 1 January of that year (`sentinel.eda.purge_table` counts them); four levels:
  naive baseline → rule → classic model → LightGBM; choose the **simplest level within one
  standard error of the best** (V2-50: the standard error of the four yearly differences from the
  best level); report every test year, not only the average.
- **Show forecasts as a chance in five bands, each with its real dilution rate on the test years** with their hit rates on the test years (D62), never as a bare
  number.
- **Reproducible:** sort rows by `cik`, `as_of` before modelling; fixed seeds (`SEED = 0`); log
  each run in MLflow with the git commit and the export date. Two runs must give identical
  predictions.
- **Private data:** outputs are stripped on commit (nbstripout); written conclusions quote
  aggregates only, never one company's prices or returns (Tiingo terms). Never write to `data/`.
- **Run it** start to finish under the memory cap:
  `systemd-run --user --scope -p MemoryMax=5G -p MemorySwapMax=0 uv run jupyter nbconvert --to notebook --execute --inplace notebooks/007-dilution-model.ipynb`
- **Every fitted step knows when it learned (V2-49).** Each model, calibration and set of band
  edges is a `sentinel.models.dated.Dated`: it learns only from rows whose answers were known
  before its cutoff (the start of the period it is judged on) and refuses to forecast rows dated
  before it. No early stopping: the number of trees is chosen by predicting the validation year.
  The folds come from `eda.folds` and are checked by `eda.check_folds`.

### Two kinds of test, and the final one is not in this notebook

1. **Test years 2019–2022 (here).** For each test year, train on the earlier years only (with
   the purge) and score that year. This is where the four levels are compared and **one is
   chosen**. Because the choice is made by looking at these results, they flatter the winner a
   little.
2. **The holdout: answers whose windows start in 2023 or later (not here, not in any notebook).**
   It answers the only question that matters in the end: does the model chosen here work on data
   nobody looked at while choosing it? That works only if it is used **once**.

**How it stays hidden.** The training file has no answers after 2022: the pipeline does not even
compute them (the label code stops at windows ending 31 December 2022). This notebook must not
read the private reference copies in `data/exports/reference/` (feature 002), which hold them.

**How the final test is run (feature 010, 16 November).**

1. This feature ends with the chosen model **frozen** in `src/sentinel/models/`: its settings,
   its training rows (to 2022), its seed, the git commit and the export date, logged in MLflow.
   The report (008) is written from the 2019–2022 results.
2. The researcher then runs **one guarded command** (built in 010). It checks that the models are
   frozen and that no holdout run exists yet, computes the 2023+ answers, scores the frozen
   models, and writes the results with a log entry. A second run is refused: the log is the proof
   that the test happened once.
3. The results go into the report as they came out. The model is not changed afterwards
   (constitution IV). Later refits (021) may use the 2023+ answers, as a logged step.

A command, not a notebook, because a notebook can be re-run with one click and nothing records
how often it was looked at.

### Make the model holdout-ready (so the one test runs without a hitch)

The holdout rows come from the same feature table as the training rows (`feature_snapshot`
from 2023 on: same columns, same order, same types, checked on 30 October) and their answers from
the same label code with a later end date. What can still break is the model itself, so:

1. **Select features by name, never by position**: save the list of feature columns with the
   model and use `rows[features]` when scoring.
2. **Every preprocessing step lives inside the model** (cut points, fill values, missing flags,
   scaling), fitted on the training rows only and saved with it, never recomputed on new rows.
3. **Save one object** (in MLflow): the fitted preprocessing and model, the feature list, the
   eligibility flag it uses (`{elig}`), the seed, the git commit and the export date.
4. **Dry run before 16 November**: load the saved model and score `latest.parquet` and the 2023+
   rows of `feature_snapshot.parquet` (features only, no answers): it must run, give one
   prediction per eligible row, no unexpected missing values, values in a plausible range. This
   looks at no outcome, so it does not spend the holdout.
5. **Scoring is a function** `predict(model, rows) -> predictions` in `src/sentinel/models/`,
   the same one used for the test years, the dry run, the holdout and the daily scores.

### What the EDA found for this model (notebooks/004-eda-dilution.ipynb, Findings)

- **Rule level ready-made:** "an offering filed in the last 12 months" (`n_offering_12m > 0`):
  AUC 0.70–0.73 in every test year; dilution 21.8% if flagged, 3.8% if not. A model must clearly
  beat it.
- Strongest features: offerings and shelf filings, share growth over 12 months, interest
  coverage (low or missing), R&D intensity, stock-based pay to revenue, volatility, low price,
  small size, loss-making. Macro and short volume add nothing (drop them from the classic
  model).
- Missing values carry meaning (no interest coverage often means a small company without debt):
  add "was missing" flags.
- **Foreign filers are left out of this question (V2-27, decided 30 Oct):** train, test and
  score only rows with `foreign_filer == False`; 78% of their rows have no answer and the rest
  are not typical.
- Rows with a stock split in the window: splits are applied where the counts show them (V2-29);
  the split-like "yes" answers that remain are dropped, found by code in section 1 (V2-39,
  V2-40). Impossible share counts (V2-23) can make wrong answers.

**Data cleaning** (V2-33): the pipeline's cleaning is frozen at V2-32 (foreign market values,
share-count typos, insider value typos, placeholder prices, splits in the dilution answer, Tiingo
instruments; `docs/decisions.md`). Any further cleaning is part of **this model**: cut extreme
values at the 1st and 99th percentiles of the training rows, drop rows with a known doubt, add
"was missing" flags; write it here, explain why, and move it into `src/sentinel/models/` with the
model so the daily scoring cleans the same way. Do not rebuild the database for small fixes.

## 1. Setup and data

The development rows (dated 2022 or earlier) with this question's flag and an answer, minus the
split-like "yes" answers (a stock split read as dilution, V2-39), which are found by code in the
database (V2-40). The features are the export's columns minus `cum_split_after` (look-ahead,
V2-38); the trees leave out the insider and macro families (V2-36), which section 9b re-checks.

In [ ]:
# Setup: libraries, the rows the model learns from, the features, MLflow
import hashlib
import re
import subprocess
import tempfile
import warnings
from datetime import date
from pathlib import Path

import duckdb
import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

from sentinel import config, db, eda
from sentinel.models import dilution  # the approved model's code (V2-52)
from sentinel.models.dated import Dated

SEED, THREADS = dilution.SEED, dilution.THREADS  # LightGBM on 8 of the 16 cores
threadpool_limits(THREADS)
warnings.filterwarnings("ignore", category=RuntimeWarning)  # nan-quantiles of empty columns
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
P = config.paths()

ANSWER, ELIG = dilution.ANSWER, dilution.ELIGIBILITY
raw = eda.load("dilution")

# The split-like "yes" answers: a stock split read as dilution (V2-39), found by code in the
# database (V2-40), never read from a file; the learnable rows leave them out
with duckdb.connect(str(P.build_db), read_only=True,
                    config={"memory_limit": db.MEMORY_LIMIT}) as con:
    split_like = dilution.split_like_answers(con)
rows = dilution.learnable(raw, split_like)
y = rows[ANSWER].astype(int)
print(f"{len(split_like)} split-like 'yes' answers; "
      f"{int((raw[ELIG] & raw[ANSWER].notna()).sum()) - len(rows)} of them were learnable rows "
      "and are dropped")

# Features: the export's 96, minus cum_split_after (product of split ratios AFTER the row's day:
# look-ahead, V2-38). Identifiers, flags, the answer and its window are never features.
LOOK_AHEAD = dilution.LOOK_AHEAD
FEATURES = [c for c in eda.features(rows) if c not in LOOK_AHEAD]
NEVER = {"dilution", "window_end", "cik", "as_of", "cum_split_after", "month_end", "release_eve",
         "latest", "coverage", "filer_type", "foreign_filer", "sector", "active_filer",
         "excluded", "eligible", "eligible_volatility", "eligible_dilution"}
assert not NEVER & set(FEATURES) and len(FEATURES) == 95
FAMILY = {c: eda.family(c) for c in FEATURES}
MACRO = [c for c in FEATURES if FAMILY[c] == "macro"]
SHORT = [c for c in FEATURES if FAMILY[c] == "short volume"]
INSIDER = [c for c in FEATURES if FAMILY[c] == "insider"]
# The trees leave out the insider and macro families (V2-36); section 9b re-checks it
TREE_FEATURES = dilution.features(rows)
assert TREE_FEATURES == [c for c in FEATURES if c not in INSIDER + MACRO]

print(f"{len(rows):,} rows, {rows.cik.nunique():,} companies, base rate {y.mean():.2%}")
print(f"{len(FEATURES)} features (left out: {LOOK_AHEAD}); trees use {len(TREE_FEATURES)} "
      f"(without {len(INSIDER)} insider and {len(MACRO)} macro)")
display(rows[["cik", "as_of", ANSWER]].assign(year=rows.as_of.dt.year).groupby("year")
        .agg(rows=("cik", "size"), companies=("cik", "nunique"), dilution_rate=(ANSWER, "mean")))

# MLflow: runs are logged to data/mlflow.db, artifacts to data/mlartifacts (research R9)
mlflow.set_tracking_uri(f"sqlite:///{P.mlflow_db}")
EXPERIMENT = "007-dilution"
if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(EXPERIMENT,
                             artifact_location=(P.mlflow_db.parent / "mlartifacts").as_uri())
mlflow.set_experiment(EXPERIMENT)
SRC = eda.EXPORT / "dilution.parquet"
TAGS = {
    "git_commit": subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                                 text=True).stdout.strip(),
    "export_date": date.fromtimestamp(SRC.stat().st_mtime).isoformat(),
    "data_hash": hashlib.sha256(SRC.read_bytes()).hexdigest()[:12],
    "level3_estimator": f"lightgbm {lgb.__version__}",
    "rules": "Dated cutoffs (V2-49); one SE of the yearly differences (V2-50)",
}
print(TAGS)

## 2. Walk-forward folds

For each test year Y the rows fall in blocks by their date and the day their answer was known
(`eda.folds`): **inner** (answer known before Y−1: the models learn from these), **gap** (dated
before Y−1, answer known during Y−1: left out), **valid** (dated in Y−1, answer known before Y:
settings, calibration and band edges), **purged** (answer known in Y or later: left out) and
**test** (dated in Y: scored once). `eda.check_folds` refuses any block out of time order; the
table shows each fold's boundaries. Every fitted step is a `Dated` with its cutoff: 1 January
of Y−1 for the models, 1 January of Y for settings, calibration and band edges.

In [ ]:
# Walk-forward folds cut by dates and checked (eda.folds, eda.check_folds), and the helpers
FOLDS = eda.folds(rows, ANSWER)
display(eda.check_folds(rows, FOLDS, ANSWER))  # refuses any block out of time order
for f in FOLDS.values():
    f["train"] = f["inner"] | f["gap"] | f["valid"]  # every answer known before the test year
fold_table = pd.DataFrame({year: {k: int(m.sum()) for k, m in f.items()}
                           for year, f in FOLDS.items()}).T
display(fold_table)
expected = eda.purge_table(rows, "window_end")
assert (fold_table[["train", "purged", "test"]].values
        == expected[["train_rows", "purged", "test_rows"]].values).all()
print("fold counts equal the EDA's purge table")


def valid_start(year: int) -> pd.Timestamp:
    """The cutoff of the models judged on year `year`: answers known before its validation year."""
    return pd.Timestamp(year - 1, 1, 1)


def test_start(year: int) -> pd.Timestamp:
    """The cutoff of the settings, calibration and band edges used on year `year`."""
    return pd.Timestamp(year, 1, 1)


def scores(y_true, p) -> dict:
    """Log loss (deciding), AUC, Brier score and top-tenth lift of chances `p`."""
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    yt, ps = pd.Series(np.asarray(y_true, dtype=float)), pd.Series(p)
    return {"log_loss": log_loss(yt, p, labels=[0, 1]), "auc": eda.auc(yt, ps),
            "brier": brier_score_loss(yt, p), "lift_top10": eda.top_decile_lift(yt, ps)}


RESULTS, TEST_PRED, TEST_RAW, VALID_RAW, FITTED = [], {}, {}, {}, {}


def record(level: str, year: int, p_test, params: dict, raw_test=None, raw_valid=None,
           extra=None) -> dict:
    """Keep a level's test-year chances and scores, and log them as one MLflow run."""
    m = scores(y[FOLDS[year]["test"]], p_test)
    RESULTS.append({"level": level, "year": year, **m})
    TEST_PRED[(level, year)] = np.asarray(p_test, dtype=float)
    TEST_RAW[(level, year)] = np.asarray(p_test if raw_test is None else raw_test, dtype=float)
    if raw_valid is not None:
        VALID_RAW[(level, year)] = np.asarray(raw_valid, dtype=float)
    with mlflow.start_run(run_name=f"{level}-{year}"):
        mlflow.set_tags({**TAGS, "level": level, "test_year": year})
        mlflow.log_params(params)
        mlflow.log_metrics({**m, **(extra or {})})
    return m


def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def calibrator(p_valid, mask, year: int) -> Dated:
    """Platt scaling: a logistic fit of the answer on the model's log-odds (research R6), on the
    validation year's rows, all known before the test year."""
    frame = rows.loc[mask, ["as_of", "window_end"]].assign(logit=logit(p_valid))
    cal = Dated(LogisticRegression(C=1e6), ANSWER, ["logit"]).fit(frame, test_start(year),
                                                                   y=y[mask])
    assert cal.estimator.coef_[0, 0] > 0  # it keeps the order of the chances (section 8)
    return cal


def calibrate(cal: Dated, p, mask) -> np.ndarray:
    """Calibrated chances for the rows of `mask`, all dated from the calibrator's cutoff on."""
    return cal.predict(rows.loc[mask, ["as_of"]].assign(logit=logit(p)))

## 3. Level 0: the naive baseline

Every row gets the same chance: the dilution rate of the rows whose answers were known before
the test year.

In [ ]:
# Level 0: the base rate of each fold's training rows, the same chance for every row
class BaseRate(BaseEstimator, ClassifierMixin):
    """The rows' dilution rate, the same chance for every row."""

    def fit(self, X, y):
        self.rate_ = float(np.mean(y))
        return self

    def predict_proba(self, X):
        p = np.full(len(X), self.rate_)
        return np.column_stack([1 - p, p])


for year, f in FOLDS.items():
    model = Dated(BaseRate(), ANSWER, []).fit(rows[f["train"]], test_start(year))
    FITTED[("base_rate", year)] = model
    record("base_rate", year, model.predict(rows[f["test"]]),
           {"rate": round(model.estimator.rate_, 6)})
display(pd.DataFrame(RESULTS).query("level == 'base_rate'").set_index("year"))

## 4. Level 1: a rule

The EDA's one-question rule: has the company filed a share offering in the last 12 months? Each
group gets its dilution rate from the rows known before the test year.

In [ ]:
# Level 1: the EDA's rule, "an offering was filed in the last 12 months" (research R4)
class OfferingRule(BaseEstimator, ClassifierMixin):
    """The dilution rate of the rows with an offering filed in the last 12 months, and of the
    others."""

    def fit(self, X, y):
        flag, y = np.asarray(X, dtype=float)[:, 0] > 0, np.asarray(y)
        self.rates_ = {"flagged": float(y[flag].mean()), "not": float(y[~flag].mean())}
        return self

    def predict_proba(self, X):
        flag = np.asarray(X, dtype=float)[:, 0] > 0
        p = np.where(flag, self.rates_["flagged"], self.rates_["not"])
        return np.column_stack([1 - p, p])


for year, f in FOLDS.items():
    model = Dated(OfferingRule(), ANSWER, ["n_offering_12m"]).fit(rows[f["train"]],
                                                                 test_start(year))
    FITTED[("rule", year)] = model
    m = record("rule", year, model.predict(rows[f["test"]]),
               {k: round(v, 6) for k, v in model.estimator.rates_.items()})
    assert 0.69 <= m["auc"] <= 0.76, m
display(pd.DataFrame(RESULTS).query("level == 'rule'").set_index("year"))

# For context only (not levels): the other simple rules' AUC per test year
context = {"offering filed in 12 months": rows.n_offering_12m > 0,
           "shelf effective in 12 months": rows.n_shelf_effective_12m > 0,
           "shares up 10%+ in 12 months": rows.share_growth_12m >= 0.10}
display(pd.DataFrame({year: {name: eda.auc(y[f["test"]], flag[f["test"]].astype(float))
                             for name, flag in context.items()}
                      for year, f in FOLDS.items()}))

## 5. Level 2: a classic model

A regularised logistic regression: one feature of each redundant pair, every column cut at its
1st and 99th percentiles and missing values filled with the median plus a "was missing" flag
(all fitted on the training rows only, V2-33), standardised. It learns from the inner rows; the
strength of the regularisation (C) is chosen by predicting the validation year, and the chances
are calibrated there.

In [ ]:
# Level 2: a regularised logistic regression with the model-side cleaning of V2-33
class Clip(BaseEstimator, TransformerMixin):
    """Cut each column at its 1st and 99th percentiles of the rows it was fitted on."""

    def __init__(self, lower=0.01, upper=0.99):
        self.lower, self.upper = lower, upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lo_ = np.nanquantile(X, self.lower, axis=0)
        self.hi_ = np.nanquantile(X, self.upper, axis=0)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lo_, self.hi_)


# Columns: no macro, no short volume; one of each |Spearman| >= 0.9 pair, chosen on the 2019
# fold's training rows only (fixed for all folds; features against features, no answer used),
# keeping the log versions.
KEEP = {"log_addv20", "log_market_cap", "log_close"}
candidates = [c for c in FEATURES if c not in MACRO + SHORT]
pairs = eda.redundant_pairs(rows[FOLDS[2019]["train"]], candidates, threshold=0.9, seed=SEED)
redundant = []
for a, b in zip(pairs.a, pairs.b):
    if a in redundant or b in redundant or (a in KEEP and b in KEEP):
        continue
    redundant.append(a if b in KEEP else b)
LINEAR = [c for c in candidates if c not in redundant]
print(f"{len(LINEAR)} columns; left out as redundant: {redundant}")


def logistic_pipeline(C: float):
    return make_pipeline(Clip(), SimpleImputer(strategy="median", add_indicator=True),
                         StandardScaler(), LogisticRegression(C=C, max_iter=3000))


def fit_logistic(f: dict, year: int, features=None, Cs=(0.01, 0.1, 1.0)) -> dict:
    """Learn from inner rows (answers known before the validation year), choose C by predicting
    the validation year, calibrate on it."""
    features = features or LINEAR
    tried = {}
    for C in Cs:
        model = Dated(logistic_pipeline(C), ANSWER, features).fit(rows[f["inner"]],
                                                                 valid_start(year))
        p = model.predict(rows[f["valid"]])
        tried[C] = (log_loss(y[f["valid"]], p), model, p)
    C = min(tried, key=lambda c: tried[c][0])
    _, model, p_valid = tried[C]
    return {"model": model, "calibrator": calibrator(p_valid, f["valid"], year),
            "params": {"C": C}, "features": features, "valid_raw": p_valid,
            "tried": {c: round(v[0], 5) for c, v in tried.items()}}


for year, f in FOLDS.items():
    fit = fit_logistic(f, year)
    FITTED[("logistic", year)] = fit
    raw_test = fit["model"].predict(rows[f["test"]])
    record("logistic", year, calibrate(fit["calibrator"], raw_test, f["test"]), fit["params"],
           raw_test, fit["valid_raw"], {"log_loss_raw": log_loss(y[f["test"]], raw_test)})
    print(year, "C tried (validation log loss):", fit["tried"], "-> C =", fit["params"]["C"])
display(pd.DataFrame(RESULTS).query("level == 'logistic'").set_index("year"))

## 6. Level 3: LightGBM

Gradient-boosted trees (D60, V2-35) on the tree features, fixed seeds: the approved model's code, `src/sentinel/models/dilution.py` (V2-52). They learn from the inner
rows; the settings and the number of trees are chosen by predicting the validation year (no
early stopping inside the fit, V2-49), and the chances are calibrated there.

In [ ]:
# Level 3: LightGBM (D60, V2-35), the approved model's code (src/sentinel/models/dilution.py): it
# learns from inner rows, chooses its settings and number of trees by predicting the validation
# year, and is calibrated there
for year, f in FOLDS.items():
    m = dilution.fit(rows, year)
    FITTED[("boosting", year)] = m
    raw_test, p_test = m.chances(rows[f["test"]])
    record("boosting", year, p_test, {**m.setting["params"], "trees": m.setting["trees"]},
           raw_test, m.valid_raw, {"log_loss_raw": log_loss(y[f["test"]], raw_test)})
    best = sorted(m.tried.items(), key=lambda kv: kv[1])[:3]
    print(year, "best settings on the validation year:",
          [(p, t, round(v, 5)) for (p, t), v in best])
display(pd.DataFrame(RESULTS).query("level == 'boosting'").set_index("year"))

## 7. Comparison and choice

Each level's measures per test year. The choice (V2-50): of the levels whose gap to the best is
within one standard error of their four yearly gaps, the simplest. A more complex level wins when
it beats a simpler one steadily, year by year, whatever the size of its lead.

In [ ]:
# Comparison of the four levels and the year-by-year one-standard-error choice (V2-50)
ORDER = ["base_rate", "rule", "logistic", "boosting"]  # simplest first
res = pd.DataFrame(RESULTS)
for measure in ["log_loss", "auc", "brier", "lift_top10"]:
    print(measure)
    display(res.pivot(index="level", columns="year", values=measure).loc[ORDER])

LL = res.pivot(index="year", columns="level", values="log_loss")[ORDER]
best = LL.mean().idxmin()
gap = LL.sub(LL[best], axis=0)  # how much worse than the best, per test year
summary = pd.DataFrame({
    "mean log loss": LL.mean(),
    "gap to the best (mean)": gap.mean(),
    "SE of the yearly gaps": gap.std() / np.sqrt(len(LL)),
    "years the best is better": (gap > 0).sum(),
}).loc[ORDER]
summary["as good as the best"] = summary["gap to the best (mean)"] <= summary["SE of the yearly gaps"]
CHOSEN = next(level for level in ORDER if summary.loc[level, "as good as the best"])
display(summary)
print("gap to the best per test year:")
display(gap.T.loc[ORDER])
print(f"best mean log loss: {best}; CHOSEN LEVEL: {CHOSEN} (the simplest level whose gap to the "
      "best is within one standard error of its yearly gaps, V2-50)")
spread = LL[best].std() / np.sqrt(len(LL))
textbook = next(level for level in ORDER if LL[level].mean() <= LL[best].mean() + spread)
print(f"for reference, the textbook version (one standard error of the best level's own yearly "
      f"scores, {spread:.4f}) would choose: {textbook}")

wide = res.pivot(index="year", columns="level", values=["log_loss", "auc"])
beats_rule = ((wide["log_loss"][CHOSEN] < wide["log_loss"]["rule"])
              & (wide["auc"][CHOSEN] > wide["auc"]["rule"]))
beats_base = res.query("level == @CHOSEN").set_index("year")["brier"] < res.query(
    "level == 'base_rate'").set_index("year")["brier"]
display(pd.DataFrame({"beats rule (log loss and AUC)": beats_rule,
                      "beats base-rate Brier": beats_base}))
print("Note: the 2022 test year covers January-June only (later windows end after 2022).")

## 8. Calibration and bands

Five bands per test year, with edges at the quintiles of the validation year's chances (fitted
before the test year: a `Dated` with the test year's cutoff). Calibration keeps the order of the
chances, so the model's own validation chances set the edges without re-scoring rows the
calibration learned from. Each band shows the average forecast and the real dilution rate in the
test year (its hit rate, D47).

In [ ]:
# Five bands with edges from the validation year, and calibration charts (research R6)
Bands = dilution.Bands  # quintiles of the validation year's chances


BAND_LEVEL = CHOSEN if CHOSEN in ("logistic", "boosting") else "boosting"
if BAND_LEVEL != CHOSEN:
    print(f"The chosen level ({CHOSEN}) gives at most two chances, so bands are shown for "
          f"{BAND_LEVEL} for comparison.")
bands, monotone = [], {}
for year, f in FOLDS.items():
    valid = rows.loc[f["valid"], ["as_of", "window_end"]].assign(
        chance=VALID_RAW[(BAND_LEVEL, year)])
    banding = Dated(Bands(), ANSWER, ["chance"]).fit(valid, test_start(year), y=y[f["valid"]])
    band = banding.predict(rows.loc[f["test"], ["as_of"]].assign(
        chance=TEST_RAW[(BAND_LEVEL, year)]))
    t = (pd.DataFrame({"band": band, "chance": TEST_PRED[(BAND_LEVEL, year)],
                       "real": y[f["test"]].to_numpy()})
         .groupby("band").agg(rows=("real", "size"), forecast=("chance", "mean"),
                              real_rate=("real", "mean")))
    monotone[year] = bool(t["real_rate"].is_monotonic_increasing)
    bands.append(t.assign(year=year).reset_index())
band_table = pd.concat(bands).pivot(index="band", columns="year",
                                    values=["rows", "forecast", "real_rate"])
display(band_table)
print("real rate rises from band 1 to band 5:", monotone)

fig, axes = plt.subplots(1, len(FOLDS), figsize=(16, 4), sharey=True)
for ax, (year, f) in zip(axes, FOLDS.items()):
    d = pd.DataFrame({"p": TEST_PRED[(BAND_LEVEL, year)], "y": y[f["test"]].to_numpy()})
    d["group"] = pd.qcut(d["p"].rank(method="first"), 10, labels=False)
    g = d.groupby("group").mean()
    ax.plot([0, g.max().max()], [0, g.max().max()], color="grey", lw=1, ls="--")
    ax.plot(g["p"], g["y"], marker="o")
    ax.set_title(f"{year}")
    ax.set_xlabel("forecast chance")
axes[0].set_ylabel("real dilution rate")
fig.suptitle(f"Calibration of {BAND_LEVEL}, ten groups per test year")
plt.show()

## 9. What the model uses

The strongest inputs of each model (2021 fold), and a leak check: no single feature may be
near-perfect on its own.

In [ ]:
# What the models use, and a leak check (research R7)
pipe = FITTED[("logistic", 2021)]["model"].estimator
lr, imputer = pipe[-1], pipe[1]
labels = LINEAR + [f"{LINEAR[i]} (was missing)" for i in imputer.indicator_.features_]
top_coef = (pd.Series(lr.coef_[0], index=labels).sort_values(key=np.abs, ascending=False)
            .head(20).rename("standardised coefficient (2021 fold)"))
display(top_coef.to_frame())

boost = FITTED[("boosting", 2021)]
trees = boost.trees.estimator
gain = pd.Series(trees.model_.booster_.feature_importance("gain", iteration=trees.trees_),
                 index=boost.features)
top_gain = (gain / gain.sum()).sort_values(ascending=False).head(20).rename(
    "share of LightGBM gain (2021 fold)")
display(top_gain.to_frame().assign(family=lambda t: [FAMILY[c] for c in t.index]))

# Leak check: inputs are features only, and no single feature is near-perfect on training rows
assert set(LINEAR) <= set(FEATURES) and set(TREE_FEATURES) <= set(FEATURES)
assert not NEVER & set(LINEAR + TREE_FEATURES)
tr19 = FOLDS[2019]["train"]
single = pd.Series({c: eda.auc(y[tr19], rows.loc[tr19, c]) for c in FEATURES
                    if rows.loc[tr19, c].notna().sum() > 1000})
suspicious = single[(single >= 0.95) | (single <= 0.05)]
print("features with single AUC >= 0.95 or <= 0.05 on 2019 training rows:", list(suspicious.index))
assert suspicious.empty

## 9b. The hypotheses (V2-46, docs/hypotheses.md)

Each hypothesis names a family of features and why it should matter. **Family check:** the chosen
level is refitted without the family (or with it added back, for families it leaves out), on every
fold with that fold's settings, five seeds; the family **adds** if log loss is worse without it,
averaged over the four test years, by more than one standard error of the four yearly
differences. **Direction check:** the rank correlation of the key feature with the answer in each
test year; the expected sign in at least three of the four years. **Verdict:** supported (adds,
right direction), not supported (adds nothing), opposite (the direction is reversed in at least
three years). H-D5 expects no effect: supported when none of its families adds.

In [ ]:

# The hypotheses for dilution (docs/hypotheses.md): family checks and direction checks
CHECK_LEVEL = CHOSEN if CHOSEN in ("logistic", "boosting") else "boosting"
BASE = TREE_FEATURES if CHECK_LEVEL == "boosting" else LINEAR
SEEDS = range(5) if CHECK_LEVEL == "boosting" else range(1)  # the logistic fit has no seed
GROUP = {
    "H-D1 offerings (filing events)": [c for c in BASE if FAMILY[c] == "filing events"],
    "H-D2 need (runway, losses, cash, R&D, stock pay)": [
        c for c in ["cash_runway_years", "loss_making", "cash_to_assets", "rd_intensity",
                    "sbc_to_revenue"] if c in BASE],
    "H-D3 habit (share growth, issue proceeds)": [
        c for c in ["share_growth_12m", "issue_proceeds_to_assets"] if c in BASE],
    "H-D4 size and price": [c for c in BASE if FAMILY[c] in ("size", "price")],
}
CHECKS = {h: (BASE, [c for c in BASE if c not in g]) for h, g in GROUP.items()}  # (with, without)
for name, fam in {"insider": INSIDER, "macro": MACRO, "short volume": SHORT}.items():
    inside = any(c in BASE for c in fam)
    CHECKS[f"H-D5 {name}"] = ((BASE, [c for c in BASE if c not in fam]) if inside
                              else (BASE + fam, BASE))


def check_log_loss(features: list, year: int, seed: int) -> float:
    """Test-year log loss of the chosen level refitted on `features` with the fold's settings."""
    f = FOLDS[year]
    if CHECK_LEVEL == "boosting":
        m = dilution.fit(rows, year, features, seed, FITTED[("boosting", year)].setting)
        return log_loss(y[f["test"]], m.chances(rows[f["test"]])[1])
    fit = fit_logistic(f, year, features, (FITTED[("logistic", year)]["params"]["C"],))
    raw = fit["model"].predict(rows[f["test"]])
    return log_loss(y[f["test"]], calibrate(fit["calibrator"], raw, f["test"]))


variants = {tuple(v) for pair in CHECKS.values() for v in pair}
CHECK_LL = {(v, year): np.mean([check_log_loss(list(v), year, s) for s in SEEDS])
            for v in variants for year in FOLDS}
family_rows = []
for h, (with_f, without_f) in CHECKS.items():
    d = pd.Series({year: CHECK_LL[(tuple(without_f), year)] - CHECK_LL[(tuple(with_f), year)]
                   for year in FOLDS})  # positive: worse without the family
    se = d.std() / np.sqrt(len(d))
    family_rows.append({"check": h, "features": len(set(with_f) ^ set(without_f)),
                        **{str(year): v for year, v in d.items()}, "mean": d.mean(), "se": se,
                        "adds": bool(d.mean() > se)})
family_table = pd.DataFrame(family_rows).set_index("check")
print(f"log loss without the family minus with it ({CHECK_LEVEL}, seeds {list(SEEDS)}): "
      "positive means the family helps")
display(family_table)

# Direction checks: rank correlation of each key feature with the answer, per test year
KEY = {"H-D1": ("n_offering_12m", +1), "H-D2": ("loss_making", +1),
       "H-D3": ("share_growth_12m", +1), "H-D4": ("log_market_cap", -1)}
test_rows = rows[np.logical_or.reduce([f["test"].to_numpy() for f in FOLDS.values()])]
rho = eda.rank_by_year(test_rows, [k for k, _ in KEY.values()], ANSWER)
display(rho)

rule_beats_base = bool((LL["rule"] < LL["base_rate"]).all())
verdicts = []
for h, (feature, sign) in KEY.items():
    check = next(c for c in family_table.index if c.startswith(h))
    right = int((np.sign(rho.loc[feature]) == sign).sum())
    wrong = int((np.sign(rho.loc[feature]) == -sign).sum())
    adds = bool(family_table.loc[check, "adds"]) and (h != "H-D1" or rule_beats_base)
    verdict = "opposite" if wrong >= 3 else ("supported" if adds and right >= 3
                                             else "not supported")
    verdicts.append({"hypothesis": check, "expected": f"{feature} {'+' if sign > 0 else '−'}",
                     "change without (mean ± SE)": f"{family_table.loc[check, 'mean']:+.4f} ± "
                                                   f"{family_table.loc[check, 'se']:.4f}",
                     "expected sign in years": f"{right} of {len(FOLDS)}", "verdict": verdict})
h5 = family_table.loc[[c for c in family_table.index if c.startswith("H-D5")]]
verdicts.append({"hypothesis": "H-D5 insider, short volume, macro", "expected": "no effect",
                 "change without (mean ± SE)": "; ".join(
                     f"{c[5:]} {r['mean']:+.4f} ± {r['se']:.4f}" for c, r in h5.iterrows()),
                 "expected sign in years": "–",
                 "verdict": "not supported" if h5["adds"].any() else "supported"})
VERDICTS = pd.DataFrame(verdicts).set_index("hypothesis")
print("H-D1 also needs the offering rule to beat the base rate in every test year:",
      rule_beats_base)
display(VERDICTS)
with mlflow.start_run(run_name="hypotheses"):
    mlflow.set_tags({**TAGS, "level": CHECK_LEVEL, "check": "hypotheses"})
    for check, r in family_table.iterrows():
        mlflow.log_metric(re.sub(r"[^A-Za-z0-9_]", "_", check)[:60] + "_mean", r["mean"])


## 10. Frozen model, dry run and findings

The chosen level made for the holdout exactly as for a test year (target year 2023): the model
learns from answers known before 2022, its settings, calibration and band edges come from the
2022 rows (known before 2023). It is one object of `Dated` steps with one `predict` function, both in
`src/sentinel/models/dilution.py` (`sentinel train dilution` makes the same model); a dry run
scores rows without answers.

In [ ]:
# The frozen model: the approved code (src/sentinel/models/dilution.py, V2-52) made for the holdout
# year exactly as for a test year: one object of Dated steps and one predict function
assert CHOSEN == "boosting", CHOSEN  # the level the researcher approved (V2-52)
made_from = {k: TAGS[k] for k in ("git_commit", "export_date", "data_hash")}
frozen = dilution.fit(rows, dilution.TARGET_YEAR, provenance=made_from)
again = dilution.fit(rows, dilution.TARGET_YEAR, provenance=made_from)
print(f"cutoffs: trees {frozen.trees.cutoff.date()}, calibration {frozen.calibrator.cutoff.date()}, "
      f"bands {frozen.bands.cutoff.date()}; the trees learned from answers through "
      f"{frozen.trained_through}; setting {frozen.setting}")

# Dry run on rows without answers: each company's newest row, and the 2023+ feature rows
latest = pd.read_parquet(eda.EXPORT / "latest.parquet")
latest["as_of"] = pd.to_datetime(latest["as_of"])
# only companies still trading, as in the daily scores (`scoring.daily.CURRENT_DAYS`, 7 days):
# a company's newest row from years ago may be one whose answer the model learned from
latest = latest[latest["as_of"] >= latest["as_of"].max() - pd.Timedelta(days=7)]
snap = pq.read_table(eda.EXPORT / "feature_snapshot.parquet",
                     filters=[("as_of", ">", eda.DEV_END)]).to_pandas()
assert ANSWER not in snap.columns and ANSWER not in latest.columns
for name, frame in {"latest (still trading)": latest, "2023+ feature rows": snap}.items():
    out, out2 = dilution.predict(frozen, frame), dilution.predict(again, frame)
    scored = out[out["eligible"]]
    eligible = int(frame[ELIG].fillna(False).astype(bool).sum())
    assert len(scored) == eligible and scored["dilution_prob"].between(0, 1).all()
    assert out["dilution_prob"].equals(out2["dilution_prob"])
    print(f"{name}: {len(frame):,} rows, {eligible:,} eligible, all scored; two fits identical: "
          f"True; bands: {dict(sorted((int(b), int(n)) for b, n in scored['dilution_band'].value_counts().items()))}")

# The guard at work: the frozen model refuses a row dated before its cutoff
try:
    dilution.predict(frozen, rows[rows.as_of.dt.year == 2021].head(1))
    raise RuntimeError("a 2021 row was scored")
except AssertionError as e:
    print("refused as expected:", str(e)[:90])

with tempfile.TemporaryDirectory() as tmp, mlflow.start_run(run_name="frozen-boosting"):
    path = Path(tmp) / "dilution_model.joblib"
    dilution.save(frozen, path)
    mlflow.set_tags({**TAGS, "level": "boosting", "frozen": "approved (V2-52)"})
    mlflow.log_params({"level": "boosting", "trained_through": frozen.trained_through,
                       "n_features": len(frozen.features),
                       "model_cutoff": str(frozen.trees.cutoff.date())})
    mlflow.log_artifact(str(path))
print(f"frozen model saved in MLflow: band edges (the trees' chances) "
      f"{np.round(frozen.bands.estimator.edges_, 4).tolist()}")

## 11. Record for the report (feature 008)

The tables above, recorded as one MLflow run named `report` in this notebook's experiment;
`sentinel report` writes the model report and the model card data from it
(`specs/008-model-report/contracts/report-record.md`). Aggregates only.

In [ ]:
# Record the report's tables (feature 008): one MLflow run named `report`, read by `sentinel report`
from sentinel.models import report

calibration = []  # ten groups per test year: mean chance and real dilution rate
for year, f in FOLDS.items():
    d = pd.DataFrame({"forecast": TEST_PRED[(BAND_LEVEL, year)], "real": y[f["test"]].to_numpy()})
    d["group"] = pd.qcut(d["forecast"].rank(method="first"), 10, labels=False) + 1
    calibration.append(d.groupby("group").mean().assign(year=year).reset_index())
record_tables = {
    "title": "Dilution", "feature": "007", "approved": "V2-52",
    "question_text": "Will this company's share count rise sharply within the next six months, "
                     "new shares diluting its shareholders?",
    "decisions": [
        "V2-35: level 3 is LightGBM, as D60 names it.",
        "V2-36: the trees leave out the insider and macro families (decided after the first "
        "version's family check, so after seeing the test years; section 9b checks them again).",
        "V2-37: the levels are chosen by log loss.",
        "V2-39, V2-40: the split-like 'yes' answers (a stock split read as dilution) are dropped, "
        "found by code.",
        "V2-49: every fitted step is a Dated; V2-50: the year-by-year one-standard-error rule.",
        "V2-52: LightGBM approved by the researcher (5 Nov).",
    ],
    "deciding": {"measure": "log_loss", "label": "log loss", "lower_is_better": True},
    "levels": ORDER, "chosen": CHOSEN, "textbook": textbook,
    "data": {"rows": len(rows), "companies": rows.cik.nunique(),
             "first row": rows.as_of.min().date(), "last row": rows.as_of.max().date(),
             "base rate": float(y.mean())},
    "folds": fold_table.rename_axis("year"),
    "results": res,
    "choice": summary.rename_axis("level"),
    "bands": pd.concat(bands),
    "calibration": pd.concat(calibration)[["year", "group", "forecast", "real"]],
    "calibration_title": "calibration of the chances (ten groups per test year)",
    "hypotheses": VERDICTS, "family_checks": family_table,
    "checks": {"yearly gaps to the best": gap.T.rename_axis("level")},
    "frozen": {"level": f"{CHOSEN} (LightGBM)", "setting": str(frozen.setting),
               "trees learned before": frozen.trees.cutoff.date(),
               "calibration and bands from": f"{frozen.year - 1} (cutoff {frozen.bands.cutoff.date()})",
               "trained through": frozen.trained_through, "features": len(frozen.features),
               "band edges (the trees' chances)": [round(float(v), 4)
                                                   for v in frozen.bands.estimator.edges_]},
}
run_id = report.record("dilution", record_tables, {**TAGS, "trial": "False"})
print("report record:", run_id)

### Findings

*Export of 5 November 2026 (data hash `d67e7d132918`; run at commit `381535c`, which imports the
approved code from `src/sentinel/models/dilution.py`, with every number identical to the run of
`d673a07`), 409,161 learnable rows of
6,299 companies (the 27 learnable split-like "yes" answers dropped, V2-39, V2-40). Features: 95
(`cum_split_after` left out, V2-38); the trees use 66, without the insider and macro families
(V2-36). Every fitted step is a `Dated` (V2-49) and the folds pass `eda.check_folds`. All numbers
are aggregates.*

**In one paragraph.** Dilution is well predictable, and machine learning earns its place: LightGBM
beats every simpler level in every test year on every measure. Its log loss is 0.225 against 0.302
for the one-question offering rule (25% lower) and 0.343 for the base rate; it ranks companies far
better (AUC 0.87–0.91 against 0.71–0.74). Its chances are honest: companies in its top band
diluted 38–46% of the time, those in its bottom band under 0.6%. It is chosen by the year-by-year
rule (V2-50) and would be by the textbook rule too. Four of the five hypotheses are supported; the
fifth fails narrowly, as short-selling volume adds a small but steady amount.

**The comparison (section 7)**

| level | log loss: mean (per year) | gap to LightGBM (mean ± SE of the yearly gaps) | AUC per test year | Brier per test year | top-tenth lift |
|---|---|---|---|---|---|
| base rate | 0.343 (0.285–0.412) | 0.118 ± 0.018 | 0.50 | 0.075–0.120 | – |
| offering rule | 0.302 (0.260–0.354) | 0.077 ± 0.011 | 0.71–0.74 | 0.072–0.108 | 1.8–2.2 |
| logistic regression | 0.249 (0.222–0.276) | 0.024 ± 0.003 | 0.85–0.89 | 0.064–0.083 | 4.2–4.9 |
| **LightGBM** | **0.225 (0.203–0.245)** | – | **0.87–0.91** | **0.059–0.075** | **4.8–5.3** |

- **Choice (V2-50):** LightGBM is better than every simpler level in each of the four test years;
  its lead over logistic regression (0.018–0.032 a year) is about nine standard errors of the yearly
  gaps, so **LightGBM** is chosen. The textbook version (one standard error of LightGBM's own
  yearly scores, 0.011) would choose it too. It beats the rule (log loss and AUC) and the base
  rate's Brier score in every test year (SC-004).
- The same as Jassin's run of 2 November on the export of 30 October (LightGBM 0.226, rule 0.302):
  the new rules (date-checked folds, `Dated` steps, no early stopping) change nothing material.
  LightGBM uses 150–300 trees of 63 leaves; logistic regression hardly depends on its
  regularisation (validation log loss equal to the fourth decimal for C from 0.01 to 1).
- The base rate's Brier score differs by year (0.075–0.120) because the dilution rate does (8–14%;
  2020 was a year of share sales). 2022 covers January–June only.

**Bands (section 8).** Edges from the validation year, fitted before each test year. In every test
year the real rate rises from band 1 to band 5 (SC-005), and the forecasts sit close to it:

| band | forecast (range over years) | real dilution rate |
|---|---|---|
| 1 | 0.4–0.6% | 0.3–0.6% |
| 2 | 1.0–1.4% | 0.7–1.3% |
| 3 | 2.2–4.0% | 1.9–3.9% |
| 4 | 7.0–13.1% | 7.7–12.7% |
| 5 | 39–47% | 38–46% |

The largest misses are in band 5: 39% forecast against 46% real in 2020, during the wave of share
sales, and 47% against 38% in 2021, the year after it. Band sizes differ from a fifth in the test
years because the edges come from the year before.

**What the model uses (section 9).** Offering filings in the last 12 and 3 months carry 29% of
LightGBM's gain (2021 fold); then idiosyncratic volatility, market value and shares outstanding
(small, volatile companies dilute more), share growth, unregistered share sales, book-to-market and
the current ratio. The logistic regression leans most on small market value. No single feature is
near-perfect on its own (no leak sign).

**Hypotheses (section 9b, V2-46).** LightGBM refitted without each family, five seeds, every fold:

| hypothesis | log loss worse without it (mean ± SE) | key feature: expected sign in the test years | verdict |
|---|---|---|---|
| H-D1 offerings: a filed offering is permission and intent to sell shares | +0.0225 ± 0.0015 | `n_offering_12m` +: 4 of 4 | supported |
| H-D2 need: losses and little cash force a company to raise money | +0.0021 ± 0.0003 | `loss_making` +: 4 of 4 | supported |
| H-D3 habit: a company that issued shares keeps issuing | +0.0020 ± 0.0002 | `share_growth_12m` +: 4 of 4 | supported |
| H-D4 size and price: small, cheap, volatile stocks dilute more | +0.0130 ± 0.0022 | `log_market_cap` −: 4 of 4 | supported |
| H-D5 insider, short volume and macro add nothing | insider +0.0000 ± 0.0002; macro −0.0024 ± 0.0020; short volume +0.0006 ± 0.0001 | – | not supported (short volume) |

In plain words: offering filings matter most by far (leaving them out costs about ten times more
than either financial group), then size and price; the financial "need" and "habit" groups add
little on top, but steadily. Insider activity adds nothing, and macro makes the model worse, which
confirms leaving both out (V2-36). Short-selling volume adds a very small but steady amount in
every year (+0.0003 to +0.0009), so H-D5 is not supported as written; a plausible mechanism, studied
in finance research, is that short selling rises before share offerings. Short volume is already
among the trees' features, so nothing changes. These verdicts are informed, not blind: the EDA
had shown the same relations on 2012–2022; the blind check is the holdout test (010).

**Holdout-ready (section 10).** The frozen LightGBM is made exactly as for a test year: it learned
from answers known by 29 December 2021 (cutoff 1 January 2022), and its settings, calibration and
band edges come from the 2022 rows (cutoff 1 January 2023). It scored all 4,104 eligible companies
still trading and all 244,818 eligible 2023+ feature rows; two independent fits gave identical
chances; it refused a 2021 row (the guard of V2-49). Saved in MLflow (experiment `007-dilution`,
run `frozen-boosting`).

**Open**

1. **Approved** by the researcher on 5 November (V2-52, SC-007).
2. **Done:** the model's code is `src/sentinel/models/dilution.py` (`fit`, `predict`, the learnable
   rows), with tests; this notebook imports it and gives the same numbers (T024–T025);
   `sentinel train dilution` makes the frozen model.
3. The holdout test (010) drops the split-like answers from the 2023+ answers by the same rule
   (FR-018).